# Fase 2.1 — Modelo SARIMAX

**Thesis:** Comparativa de algoritmos para la predicción de la demanda eléctrica  
**Author:** Antonio Navarro  

This notebook:
1. Loads the feature dataset produced in Fase 1
2. Selects exogenous variables for SARIMAX
3. Uses `pmdarima.auto_arima` to find the best (p,d,q)(P,D,Q,24) order on the training set
4. Performs a rolling 24-hour-ahead forecast on the test set
5. Computes MAPE, RMSE, MAE and saves predictions to `data/predictions_sarimax.csv`

**Note:** SARIMAX with s=24 and several years of hourly data is computationally heavy.
The grid search uses `stepwise=True` to keep fit time manageable.

In [1]:
# Install dependencies not pre-installed in Colab
import importlib, subprocess, sys

def _pip(pkg, import_name=None):
    if importlib.util.find_spec(import_name or pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_pip("pmdarima")
_pip("statsmodels")
print("Dependencies ready.")

Dependencies ready.


In [2]:
import sys, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings

from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pmdarima as pm

# ── Colab: clone or update the repo, then cd into it ──────────────────────────
REPO_NAME = "pred_demanda"
REPO_URL  = "https://github.com/antonionc/pred_demanda.git"

if os.path.exists("data_utils.py"):
    !git pull
elif os.path.isdir(REPO_NAME):
    %cd {REPO_NAME}
    !git pull
elif os.path.exists(f"/content/{REPO_NAME}"):
    %cd /content/{REPO_NAME}
    !git pull
elif os.path.exists("/content"):
    !git clone {REPO_URL}
    %cd {REPO_NAME}
# ──────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, os.path.abspath('.'))
import data_utils as du

# Mount Google Drive in Colab and create persistent links for data/, cache/, saved_models/
du.setup_colab_drive()

warnings.filterwarnings('ignore')
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})

print('Environment ready.')

  [DriveSync] Downloading data/fig_ttm_fewshot_week.png...
  [DriveSync] Downloading data/fig_boxcox.png...
  [DriveSync] Downloading data/fig_lstm_training.png...
  [DriveSync] Downloading data/fig_lstm_residuals.png...
  [DriveSync] Downloading data/fig_acf_pacf.png...
  [DriveSync] Downloading data/fig_seasonality.png...
  [DriveSync] Downloading data/predictions_lstm.csv...
  [DriveSync] Downloading data/metrics_ttm_zeroshot.json...
  [DriveSync] Downloading data/hourly_features_full.csv...
  [DriveSync] Downloading data/fig_lstm_test.png...
  [DriveSync] Downloading data/fig_ttm_finetune.png...
  [DriveSync] Downloading data/metrics_sarimax.json...
  [DriveSync] Downloading data/fig_sarimax_residuals.png...
  [DriveSync] Downloading data/predictions_ttm_zeroshot.csv...
  [DriveSync] Downloading data/metrics_ttm_fewshot.json...
  [DriveSync] Downloading data/fig_raw_demand.png...
  [DriveSync] Downloading data/fig_stl_decomposition.png...
  [DriveSync] Downloading data/fig_ttm_comp

## 1 · Load data

In [3]:
df_train = pd.read_csv('data/features_train.csv', parse_dates=['datetime'])
df_val   = pd.read_csv('data/features_val.csv',   parse_dates=['datetime'])
df_test  = pd.read_csv('data/features_test.csv',  parse_dates=['datetime'])

print(f'Train: {len(df_train):,} rows  ({df_train["datetime"].min().date()} → {df_train["datetime"].max().date()})')
print(f'Val  : {len(df_val):,} rows  ({df_val["datetime"].min().date()} → {df_val["datetime"].max().date()})')
print(f'Test : {len(df_test):,} rows  ({df_test["datetime"].min().date()} → {df_test["datetime"].max().date()})')

Train: 29,259 rows  (2020-01-08 → 2023-05-11)
Val  : 9,753 rows  (2023-05-11 → 2024-06-20)
Test : 9,754 rows  (2024-06-20 → 2025-07-31)


## 2 · Select exogenous variables

In [4]:
# Exogenous features for SARIMAX — keep the set small to avoid excessive computation
EXOG_COLS = [
    # Calendar cyclical
    'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos',
    # Calendar flags
    'is_weekday', 'is_saturday', 'is_sunday', 'is_national_holiday',
    # Temperature (representative cities)
    'MAD_temperature_2m', 'BCN_temperature_2m', 'VLC_temperature_2m',
    'SEV_temperature_2m', 'ZGZ_temperature_2m', 'MLG_temperature_2m',
    # HDD/CDD
    'hdd', 'cdd',
]
EXOG_COLS = [c for c in EXOG_COLS if c in df_train.columns]
print(f'Using {len(EXOG_COLS)} exogenous variables:')
print(EXOG_COLS)

# Scale exogenous variables (fitted on train only)
scaler_exog = StandardScaler()
X_train_exog = scaler_exog.fit_transform(df_train[EXOG_COLS])
X_val_exog   = scaler_exog.transform(df_val[EXOG_COLS])
X_test_exog  = scaler_exog.transform(df_test[EXOG_COLS])

y_train = df_train['demand_mw'].values
y_val   = df_val['demand_mw'].values
y_test  = df_test['demand_mw'].values

print(f'\ny_train shape : {y_train.shape}')
print(f'X_train shape : {X_train_exog.shape}')

Using 16 exogenous variables:
['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekday', 'is_saturday', 'is_sunday', 'is_national_holiday', 'MAD_temperature_2m', 'BCN_temperature_2m', 'VLC_temperature_2m', 'SEV_temperature_2m', 'ZGZ_temperature_2m', 'MLG_temperature_2m', 'hdd', 'cdd']

y_train shape : (29259,)
X_train shape : (29259, 16)


## 3 · Auto-ARIMA order selection

`auto_arima` searches for the best (p,d,q)(P,D,Q,s=167) by AIC using a stepwise approach.
The search is limited to relatively small p,q values (≤24) to keep it tractable with hourly data.

In [6]:
# Use the first ~30 days of training data for fast order selection
# (full train would take hours; order generalises well)
SEARCH_ROWS = 24 * 30   # 30 days

print(f'Running auto_arima on {SEARCH_ROWS} rows ({SEARCH_ROWS//24} days)…')
t0 = time.time()

auto_model = pm.auto_arima(
    y_train[:SEARCH_ROWS],
    exogenous    = X_train_exog[:SEARCH_ROWS],
    start_p=1, max_p=24,
    start_q=0, max_q=24,
    d=None,           # let auto_arima pick
    start_P=0, max_P=1,
    start_Q=0, max_Q=1,
    D=1,               # seasonal differencing
    m=168,             # hourly data with weekly seasonality
    seasonal=True,
    stepwise=True,
    information_criterion='aic',
    error_action='ignore',
    suppress_warnings=True,
    n_fits=20,
)

elapsed = time.time() - t0
print(f'auto_arima completed in {elapsed:.1f}s')
print(f'Best order           : {auto_model.order}')
print(f'Best seasonal order  : {auto_model.seasonal_order}')
print(f'AIC                  : {auto_model.aic():.2f}')

ORDER          = auto_model.order
SEASONAL_ORDER = auto_model.seasonal_order

Running auto_arima on 720 rows (30 days)…


: 

: 

: 

## 4 · Fit SARIMAX on full training set

In [ ]:
print(f'Fitting SARIMAX{ORDER}x{SEASONAL_ORDER} on {len(y_train):,} training rows…')
t0 = time.time()

sarimax_model = SARIMAX(
    y_train,
    exog           = X_train_exog,
    order          = ORDER,
    seasonal_order = SEASONAL_ORDER,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
sarimax_fit = sarimax_model.fit(disp=False, maxiter=200)

train_time = time.time() - t0
print(f'Training completed in {train_time:.1f}s  ({train_time/60:.1f} min)')
print(sarimax_fit.summary())

## 5 · Rolling 24-hour-ahead forecast on test set

For each test day we use the fitted model parameters (no refit) and call `predict`
for the next 24 steps.  This matches the thesis evaluation protocol.

In [ ]:
# Extend the fitted model to include validation data as history
print('Applying model to validation set (updating state)…')
sarimax_val = sarimax_fit.apply(
    endog = y_val,
    exog  = X_val_exog,
    refit = False,
)

# Rolling forecast on test set — one 24-step block per day
HORIZON = 24
n_test  = len(y_test)
n_days  = n_test // HORIZON

y_pred_all  = []
y_true_all  = []
dates_all   = []

# Use the state at end of validation as starting point for test
current_fit = sarimax_val

print(f'Rolling forecast: {n_days} days × {HORIZON}h …')
t0 = time.time()

for day in range(n_days):
    start = day * HORIZON
    end   = start + HORIZON

    # Exogenous for the forecast window
    exog_fc = X_test_exog[start:end]
    fc      = current_fit.forecast(steps=HORIZON, exog=exog_fc)

    y_pred_all.extend(fc.tolist())
    y_true_all.extend(y_test[start:end].tolist())
    dates_all.extend(df_test['datetime'].iloc[start:end].tolist())

    # Update state with actual values of this window
    current_fit = current_fit.apply(
        endog = y_test[start:end],
        exog  = exog_fc,
        refit = False,
    )

    if (day + 1) % 30 == 0 or day == n_days - 1:
        print(f'  Day {day+1}/{n_days}')

inference_time = time.time() - t0
print(f'Inference completed in {inference_time:.1f}s')

y_pred = np.array(y_pred_all)
y_true = np.array(y_true_all)
dates  = pd.to_datetime(dates_all)

## 6 · Metrics

In [ ]:
metrics = du.compute_metrics(y_true, y_pred, label='SARIMAX')
metrics['train_s']     = train_time
metrics['inference_s'] = inference_time
print(f"\nTraining time  : {train_time:.1f}s")
print(f"Inference time : {inference_time:.1f}s")

## 7 · Plots

In [ ]:
# Full test-set comparison
du.plot_predictions(
    y_true, y_pred,
    title     = 'SARIMAX — actual vs predicted (test set)',
    dates     = dates,
    save_path = 'data/fig_sarimax_test.png',
)

In [ ]:
# Zoom into first week of test set
idx = slice(0, 168)
du.plot_predictions(
    y_true[idx], y_pred[idx],
    title     = 'SARIMAX — first week of test set',
    dates     = dates[idx],
    save_path = 'data/fig_sarimax_week.png',
)

In [ ]:
# Error distribution
errors = y_true - y_pred
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(errors, bins=60, color='steelblue', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('SARIMAX — residual distribution')
axes[0].set_xlabel('Error (MW)')

axes[1].plot(dates, errors, linewidth=0.5, color='steelblue')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('SARIMAX — residuals over time')
axes[1].set_ylabel('Error (MW)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.tight_layout()
plt.savefig('data/fig_sarimax_residuals.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 · Save predictions

In [ ]:
df_preds = pd.DataFrame({
    'datetime':  dates,
    'y_true':    y_true,
    'y_pred':    y_pred,
})
df_preds.to_csv('data/predictions_sarimax.csv', index=False)
print(f'Saved {len(df_preds):,} rows → data/predictions_sarimax.csv')

# Save metrics for Fase 3
import json
with open('data/metrics_sarimax.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics → data/metrics_sarimax.json')
print(metrics)

# Sync outputs to Google Drive (if in Colab)
du.sync_to_drive()


## Summary

| Metric | Value |
|--------|-------|
| MAPE   | … % |
| RMSE   | … MW |
| MAE    | … MW |
| Training time | … s |
| Inference time | … s |

Next step → `fase2_2_lstm.ipynb`